In [1]:
from decimal import Decimal

from typed_kucoin import KuCoin
from typed_kucoin.account.deposit.address import DepositAddressV3Row
from dotenv import load_dotenv

from tribulnation.sdk.wallet.deposit_methods import DepositMethod
from tribulnation.sdk.wallet.withdrawal_methods import WithdrawalMethod

load_dotenv()

client = await KuCoin.new().__aenter__()

ASSETS = ['BTC', 'ETH', 'USDT', 'KCS', 'SOL']

## `DepositMethods`

`spot.all_currencies()` is public (no credentials needed) and lists every currency
KuCoin supports along with its `chains`, one entry per blockchain network -- exactly the
per-network deposit/withdrawal parameters `DepositMethod`/`WithdrawalMethod` need.
Deposits are free on KuCoin (only withdrawals carry a fee), so `fee` is always `None`
here.

In [2]:
async def deposit_methods(*, assets: list[str] | None = None) -> list[DepositMethod]:
  currencies = await client.spot.all_currencies()
  out: list[DepositMethod] = []
  for c in currencies:
    if assets is not None and c['currency'] not in assets:
      continue
    for chain in c['chains'] or []:
      if not chain['isDepositEnabled']:
        continue
      out.append(
        DepositMethod(
          asset=c['currency'],
          network=chain['chainId'],
          fee=None,
          contract_address=chain['contractAddress'] or None,
          min_confirmations=chain['confirms'],
        )
      )
  return out


await deposit_methods(assets=ASSETS)

[DepositMethod(asset='SOL', network='sol', fee=None, contract_address=None, min_confirmations=200),
 DepositMethod(asset='BTC', network='btc', fee=None, contract_address=None, min_confirmations=3),
 DepositMethod(asset='BTC', network='bsc', fee=None, contract_address='0x7130d2a12b9bcbfae4f2634d864a1ee1ce3ead9c', min_confirmations=60),
 DepositMethod(asset='BTC', network='kcc', fee=None, contract_address='0xfa93c12cd345c658bc4644d1d4e1b9615952258c', min_confirmations=20),
 DepositMethod(asset='BTC', network='bech32', fee=None, contract_address=None, min_confirmations=2),
 DepositMethod(asset='ETH', network='bsc', fee=None, contract_address='0x2170ed0880ac9a755fd29b2688956bd959f933f8', min_confirmations=60),
 DepositMethod(asset='ETH', network='optimism', fee=None, contract_address=None, min_confirmations=500),
 DepositMethod(asset='ETH', network='eth', fee=None, contract_address=None, min_confirmations=64),
 DepositMethod(asset='ETH', network='kcc', fee=None, contract_address='0xf55af13

## `WithdrawalMethods`

Same `all_currencies()` call; `withdrawalMinFee` becomes the `Fee`, and `networks` is an
extra client-side filter on `chainId` (KuCoin doesn't take it as a request parameter --
there's no per-network lookup, only per-currency).

In [3]:
async def withdrawal_methods(
  *,
  assets: list[str] | None = None,
  networks: list[str] | None = None,
) -> list[WithdrawalMethod]:
  currencies = await client.spot.all_currencies()
  out: list[WithdrawalMethod] = []
  for c in currencies:
    if assets is not None and c['currency'] not in assets:
      continue
    for chain in c['chains'] or []:
      if networks is not None and chain['chainId'] not in networks:
        continue
      if not chain['isWithdrawEnabled']:
        continue
      out.append(
        WithdrawalMethod(
          asset=c['currency'],
          network=chain['chainId'],
          fee=WithdrawalMethod.Fee(
            asset=c['currency'], amount=Decimal(chain['withdrawalMinFee'])
          ),
          contract_address=chain['contractAddress'] or None,
        )
      )
  return out


await withdrawal_methods(assets=ASSETS)

[WithdrawalMethod(asset='SOL', network='sol', fee=WithdrawalMethod.Fee(asset='SOL', amount=Decimal('0.008')), contract_address=None),
 WithdrawalMethod(asset='BTC', network='btc', fee=WithdrawalMethod.Fee(asset='BTC', amount=Decimal('0.00009')), contract_address=None),
 WithdrawalMethod(asset='BTC', network='bsc', fee=WithdrawalMethod.Fee(asset='BTC', amount=Decimal('0.000004')), contract_address='0x7130d2a12b9bcbfae4f2634d864a1ee1ce3ead9c'),
 WithdrawalMethod(asset='BTC', network='kcc', fee=WithdrawalMethod.Fee(asset='BTC', amount=Decimal('0.00002')), contract_address='0xfa93c12cd345c658bc4644d1d4e1b9615952258c'),
 WithdrawalMethod(asset='ETH', network='bsc', fee=WithdrawalMethod.Fee(asset='ETH', amount=Decimal('0.00024')), contract_address='0x2170ed0880ac9a755fd29b2688956bd959f933f8'),
 WithdrawalMethod(asset='ETH', network='optimism', fee=WithdrawalMethod.Fee(asset='ETH', amount=Decimal('0.0002')), contract_address=None),
 WithdrawalMethod(asset='ETH', network='eth', fee=WithdrawalM

In [4]:
# Same call, narrowed to one network -- e.g. only TRC20 USDT/USDT-adjacent withdrawal methods.
await withdrawal_methods(assets=['USDT', 'BTC'], networks=['trx'])

[WithdrawalMethod(asset='USDT', network='trx', fee=WithdrawalMethod.Fee(asset='USDT', amount=Decimal('1.99')), contract_address='TR7NHqjeKQxGTCi8q8ZY4pL8otSzgjLj6t')]

### Coverage assessment: `Wallet`

**Fully supported.** `spot.all_currencies()` is public, unauthenticated, and returns the
complete per-network deposit/withdrawal configuration (enabled flags, min confirmations,
contract address, withdrawal fee) for every currency in one call -- no account-specific
state is needed to answer either `deposit_methods()` or `withdrawal_methods()`, and no
pagination or chunking is required. The one caveat: KuCoin reports a flat
`withdrawalMinFee` per network rather than a fee schedule, so `WithdrawalMethod.fee` is
the *minimum* fee, not necessarily the fee a specific withdrawal will actually be charged
(`account.withdrawals.quotas` returns the same `withdrawMinFee` figure, confirming this
is the number KuCoin itself treats as authoritative, not just a floor).

## Account-scoped supplementary reads (not part of the abstract interface)

`account.deposit.address` is authenticated and account-scoped (existing generated
addresses only), unlike the public per-network data above -- included here as a
real-data supplement, not because the SDK interface asks for it.

In [5]:
async def deposit_addresses(currency: str) -> list[DepositAddressV3Row]:
  return await client.account.deposit.address(currency=currency)


await deposit_addresses('USDT')

[{'address': '0x9efba47488f7e32ce2856994dcddc13ed3939ac4',
  'memo': '',
  'remark': '',
  'chainId': 'bsc',
  'to': 'MAIN',
  'expirationDate': datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
  'currency': 'USDT',
  'contractAddress': '0x55d398326f99059ff775485246999027b3197955',
  'chainName': 'BEP20'},
 {'address': '0x14b172f82f37202cfdb76134c3f9899fa14457c3',
  'memo': '',
  'remark': '',
  'chainId': 'eth',
  'to': 'MAIN',
  'expirationDate': datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
  'currency': 'USDT',
  'contractAddress': '0xdac17f958d2ee523a2206206994597c13d831ec7',
  'chainName': 'ERC20'}]

## Withdraw (state-mutating -- written, never executed)

`account.withdrawals.withdraw` isn't part of `WithdrawalMethods` (the abstract interface
is read-only), but is the natural next step. Included only to show the mapping --
**never executed**.

In [ ]:
async def withdraw(*, asset: str, network: str, address: str, amount: Decimal) -> str:
  result = await client.account.withdrawals.withdraw(
    currency=asset,
    to_address=address,
    amount=amount,
    withdraw_type='ADDRESS',
    chain=network,
  )
  return result['withdrawalId']


# Not executed here -- would send a real on-chain withdrawal from the account.
await withdraw(
  asset='USDT', network='trx', address='<destination-address>', amount=Decimal('10')
)